# Visualizador de Resultados e Monitor de Iterações — Benchmark F.O.C.A.

### ⚠️ Análise de Concorrência e Isolamento de Recursos
O script de benchmark (`executar_benchmark_otimizado.sh`) executa o algoritmo Fuzzy C-Means/K-Means com afinidade estrita no **Core 6 (CPU 13)**, enquanto o thread irmão SMT (CPU 12) foi colocado em estado *offline* para eliminar concorrência nos caches L1d, L1i e L2.

Embora outros núcleos estejam livres (CPUs 0 a 11), executar processamento computacional pesado dentro deste notebook durante o experimento geraria:
1. **Aquecimento do encapsulamento (Package TDP / STAPM):** Forçando *thermal throttling* e redução do clock fixo de 1.80 GHz na CPU 13.
2. **Contenção no barramento de memória (DDR4):** Interferindo na latência de acesso aos pixels da imagem.

**Diretriz:** Este notebook foi projetado **exclusivamente para monitorar e visualizar os dados gerados**, sem realizar cálculos pesados nem competir com o experimentador.

In [1]:
import os
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, clear_output

print("Ambiente de visualização pronto.")

Ambiente de visualização pronto.


## 1. Status do Processo e Isolamento de Hardware
Verifica se o benchmark está em execução em segundo plano e inspeciona o estado do clock e SMT.

In [2]:
# Lê o PID ativo do arquivo
pid_file = "benchmark.pid"
if os.path.exists(pid_file):
    with open(pid_file) as f:
        pid = f.read().strip()
    print(f"[PID Ativo Registrado]: {pid}")
    os.system(f"ps -p {pid} -o pid,user,%cpu,%mem,etime,cmd")
else:
    print("Nenhum processo de benchmark registrado no momento.")

# Inspeciona o estado do clock da CPU 13 e do Turbo Boost
try:
    with open("/sys/devices/system/cpu/cpufreq/boost") as f:
        boost = f.read().strip()
    with open("/sys/devices/system/cpu/cpu13/cpufreq/scaling_cur_freq") as f:
        freq = f.read().strip()
    with open("/sys/devices/system/cpu/cpu12/online") as f:
        sib = f.read().strip()
    print(f"\n[Estado do Hardware]:")
    print(f" - Turbo Boost: {'Desativado (0)' if boost == '0' else f'Ativado ({boost})'}")
    print(f" - CPU 13 Frequência: {int(freq)/1000:.2f} MHz")
    print(f" - CPU 12 (Irmão SMT): {'Offline (0)' if sib == '0' else f'Online ({sib})'}")
except Exception as e:
    print(f"Aviso ao ler sysfs: {e}")

[PID Ativo Registrado]: 221550
    PID USER     %CPU %MEM     ELAPSED CMD

[Estado do Hardware]:
 - Turbo Boost: Ativado (1)
 - CPU 13 Frequência: 4341.31 MHz
 - CPU 12 (Irmão SMT): Online (1)


## 2. Monitoramento em Tempo Real dos Logs de Execução
Lê as iterações e o progresso do log gerado em `logs/benchmark_*.log`.

In [3]:
logs = sorted(glob.glob("logs/benchmark_*.log"), key=os.path.getmtime, reverse=True)

if logs:
    latest_log = logs[0]
    print(f"Lendo arquivo de log mais recente: {latest_log}\n")
    with open(latest_log, 'r') as f:
        linhas = f.readlines()
    
    # Exibe as últimas 25 linhas do log
    print("".join(linhas[-25:]))
else:
    print("Nenhum log de benchmark encontrado na pasta 'logs/'.")

Lendo arquivo de log mais recente: logs/benchmark_20261020_182731.log

Convergiu em 24 iterações
Convergiu em 24 iterações
Convergiu em 10 iterações
Convergiu em 22 iterações
Convergiu em 20 iterações
Convergiu em 20 iterações
Convergiu em 10 iterações
Convergiu em 10 iterações
Convergiu em 22 iterações
Convergiu em 20 iterações
Convergiu em 10 iterações
Convergiu em 13 iterações
Convergiu em 23 iterações
Convergiu em 22 iterações
Análise de image4.jpg concluída. Memória liberada.
Carregando image1.jpeg para a RAM (n=199809)...
Traceback (most recent call last):
  File "/home/s-cesar/codes/softwareEngineerDocs/docs/samuca_files/scripts/experimentador_EP_2026.2.py", line 155, in <module>
    with Image.open(filepath) as img:
         ^^^^^^^^^^^^^^^^^^^^
  File "/home/s-cesar/codes/softwareEngineerDocs/.venv/lib/python3.12/site-packages/PIL/Image.py", line 3639, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such fil

## 3. Tabela de Estabilidade F.O.C.A. (`resultados_foca_aluno.csv`)
Carrega o arquivo CSV exportado pelo experimentador contendo média, desvio padrão e verificação de $CV < 0.15$.

In [4]:
csv_path = "resultados_foca_aluno.csv"

if os.path.exists(csv_path):
    df_resumo = pd.read_csv(csv_path, sep=';', decimal=',')
    print(f"Tabela de Calibração F.O.C.A. carregada ({len(df_resumo)} combinações):")
    display(df_resumo)
else:
    print(f"Arquivo '{csv_path}' ainda não foi gerado. O benchmark pode estar processando as primeiras repetições.")

Tabela de Calibração F.O.C.A. carregada (12 combinações):


,n,k,tempo_medio,desvio,contagem_valida,CV,estavel_CV_menor_15,tempo_predito
0,4096,2,0.888528,0.154807,26,0.174228,False,2.091250
1,4096,3,3.008372,0.689317,26,0.229133,False,3.627096
2,4096,4,2.586276,0.425440,26,0.164499,False,5.162941
3,16384,2,3.469863,0.756962,26,0.218153,False,4.563704
4,16384,3,10.720893,1.834573,26,0.171121,False,9.050331
5,16384,4,12.254915,3.382061,26,0.275976,False,13.536957
6,65536,2,13.072265,1.866527,26,0.142785,True,14.453520
7,65536,3,38.121391,5.296942,26,0.138949,True,30.743271
8,65536,4,47.690326,11.701776,26,0.245370,False,47.033022
9,262144,2,48.355426,6.415214,26,0.132668,True,54.012786


## 4. Gráficos Gerados: Curvas Bidimensionais e Superfície 3D
Exibe as imagens salvas pelo experimentador após o término dos ensaios.

In [ ]:
fig2d_path = "../outputs/relatorio_complexidade2.png"
fig3d_path = "superficie_3d_foca.png"

print("--- Corte 2D: Tempo vs n e Tempo vs K ---")
if os.path.exists(fig2d_path):
    display(Image(filename=fig2d_path))
else:
    print(f"Figura '{fig2d_path}' ainda não disponível.")

print("\n--- Superfície de Resposta 3D: T(n, K) ---")
if os.path.exists(fig3d_path):
    display(Image(filename=fig3d_path))
else:
    print(f"Figura '{fig3d_path}' ainda não disponível.")